In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.impute import SimpleImputer
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelBinarizer
import shap
from itertools import product
import plotly.graph_objects as go
from minisom import MiniSom
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import OPTICS

# Dane

In [ ]:
data=pd.read_csv("data\\financial_loan.csv")

In [ ]:
data.head()

In [ ]:
data.columns

## Opis kolumn
- id - identyfikator klienta/wniosku
- address_state - stan zamieszkania
- application_type - typ aplikacji **(zawiera tylko wartość INDIVIDUAL)**
- emp_length - długość zatrudnienia w latach
- emp_title - stanowisko prcay
- grade - ocena kredytowa **(A-G im wyższa tym lepiej)**
- home_ownership - status posiadania domu **(RENT, MORTAGE, OWN, OTHER, NONE)**
- issue_date - data wydania pożyczki
- last_credit_pull_date - data ostatniego sprawdzenia historii kredytowej
- last_payment_date - data ostatniej wpłaty na pożyczkę
- loan_status - status pożyczki **(Fully_Paid, Charged Off, Current)**
- next_payment_date - planowana data następnej wpłaty
- member_id - identyfikator klienta
- purpose - cel pożyczki **(13 kategorii plus other)**
- sub_grade - podkategoria oceny kredytowej **(bardziej dokładna niż grade, typu A1, A2, ...)**
- term - okres spłaty pożyczki **(36 months, 60 months)**
- verification_status - status weryfikacji dochodu klienta **(Not Verified, Verified, Source Verified)**
- annual_income - roczny dochód klienta
- dti (dent-to-income-ratio) - wskaźnik zadłużenia do dochodu
- installment - miesięczna rata
- int_rate - oprocentowanie
- loan_amount - kwota pożyczki
- total_acc - łączna liczba kont kredytowych klienta'
- total_payment - łączna suma spłat dokonanych przez klienta

## Cel biznesowy 
- Wykrywanie wzorców niespłacania w celu możliwości wykorzystania ich do poprawy procesów windykacyjnych przez zrozumienie charakterystyk klastrów o wysokim ryzyku.

W związku z wyborem celu biznesowego pozbywany się poniższych kolumn.

In [ ]:
data = data.drop(['int_rate', 'grade', 'sub_grade'], axis=1)

## Analiza

In [ ]:
data.info() #none pojawia się w kolumnie emp_title

Jedynie kolumna emp_title zawiera none, które mogą świadczyć o brKu zatrudnienia danej osoby.

In [ ]:
data.describe()

In [ ]:
data.nunique()

In [ ]:
columns = ['application_type', 'emp_length',
         'home_ownership', 'loan_status',
       'purpose', 'term', 'verification_status']
for col in columns:
    print(f"Unique values in {col}:")
    print(data[col].value_counts())
    print("\n")

In [ ]:
mask = data['emp_title'].map(data['emp_title'].value_counts()) > 34
data.loc[mask, 'emp_title'].value_counts()

In [ ]:
sum(data['emp_title'].value_counts()==1) #tyle nazw stanowisk pojawia się tylko raz

In [ ]:
np.divide(data['emp_title'].isnull().sum(), data['emp_title'].count()) #braki danych prawie 4%


Kolumna emp_title zawiera bardzo dużo różnych nazw, stąd jedynie wykorzystamy informację o tym czy ktoś jest zatrudniony czy nie.

In [ ]:
data['is_employed'] = np.where(data['emp_title'].isnull(), 0, 1)
data = data.drop('emp_title', axis=1)
data.head()


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(data.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm')
plt.title('Macierz korelacji')
plt.show()

In [ ]:
sns.pairplot(data)
plt.show()

# Usuwanie kolumn, pozbycie się outlierów i normalizacja

In [ ]:
data = data.drop(['id', 'member_id', 'application_type'], axis=1)

id i member_id to zmienne unikalne z założenia, a application_type przyjmuje w tym przypadku tylko jedną wartość

In [ ]:
data.columns

Usuwamy również zmienne mocno skorelowane ze sobą.

In [ ]:
data = data.drop(['installment', 'total_payment'], axis=1) #XXX ważne do późniejszego testowania

In [ ]:
sns.boxplot(x=data['loan_amount'])
plt.show()
sns.boxplot(x=data['annual_income'])
plt.show()
sns.boxplot(x=data['dti'])
plt.show()
sns.boxplot(x=data['loan_amount'])
plt.show()
sns.boxplot(x=data['total_acc'])
plt.show()


In [ ]:

data['loan_amount'] = winsorize(data['loan_amount'], limits=[0, 0.04])
sns.boxplot(x=data['loan_amount'])
plt.show()
data['annual_income'] = np.log1p(data['annual_income'])
sns.boxplot(x=data['annual_income'])
plt.show()
data['total_acc'] = winsorize(data['total_acc'], limits=[0, 0.02])
sns.boxplot(x=data['total_acc'])
plt.show()


In [ ]:
numeric_cols = data.select_dtypes(include=['int64', 'float64', 'int32']).columns
scaler = StandardScaler()
data[numeric_cols] = scaler.fit_transform(data[numeric_cols])
sns.boxplot(x=data['loan_amount'])
plt.show()
sns.boxplot(x=data['annual_income'])
plt.show()
sns.boxplot(x=data['dti'])
plt.show()
sns.boxplot(x=data['loan_amount'])
plt.show()
sns.boxplot(x=data['total_acc'])
plt.show()

# Kodowanie zmiennych

Zmienne kategoryczne: (one hot encoding z pominięciem jednej kolumny, zgrupowanie mniej popularnych kategorii do kategorii other, podział tematyczny na bardziej ogólne grupy)
- verification_status
- purpose
- loan_status
- address_state
- home_ownership

Zmienne porządkowe: (ordinal encoding + normalizacjia, label encoding)
- emp_length

Zmienne binarne: (binary encoding) 
- term
- is_employed

Zmienne numeryczne: (winsoryzacja, normalizacja)
- annual_income
- dti
- loan_amount
- total_acc

Zmienne reprezentujące daty:
- issue_date
- last_credit_pull_date
- last_payment_date
- next_payment_date

Dla dat możliwość liczenia odległości pomiędzy różnymi datami lub stworzenie zmiennych typu pora roku, kwartał, tydzień roku, dzień tygodnia, czy jest to weekend lub inne możliwie istotne dla tematu.

## Przetwarzanie dat

In [ ]:
dd = data['last_credit_pull_date']-data['issue_date']
dd_diff=[]
for i in dd:
    dd_diff.append(float(str(i).split(" ")[0]))
data['date_diff']=pd.Series(dd_diff) #różnica dni pomiędzy wydaniem kredytu a sprawdzeniem historii kredytowej

data['date_diff'] = data['date_diff'].astype(float)

data['date_diff'] = scaler.fit_transform(data[['date_diff']])

In [ ]:
data = data.drop(['issue_date', 'last_credit_pull_date', 'last_payment_date', 'next_payment_date'], axis=1)
print(data['issue_month'].unique())
print(data['issue_day'].unique())

## Kodowanie

In [ ]:
emp_length_order = ['< 1 year', '1 year', '2 years', '3 years', '4 years', 
                    '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years']

ord_encoder = OrdinalEncoder(categories=[emp_length_order])
data[['emp_length']] = ord_encoder.fit_transform(data[['emp_length']])

In [ ]:
columns = ['home_ownership', 'loan_status',
       'purpose', 'verification_status', 'purpose']
for col in columns:
    print(f"Unique values in {col}:")
    print(data[col].value_counts())
    print("\n")

In [ ]:
slownik = {
    'położenie': {
        'Northeast': ['CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA'],
        'Midwest': ['IL', 'IN', 'MI', 'OH', 'WI', 'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD'],
        'South': ['DE', 'DC', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'WV', 'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX'],
        'West': ['AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY', 'AK', 'CA', 'HI', 'OR', 'WA']
    },
    'zamożność': {
        'high': ['DC', 'MD', 'MA', 'CA', 'CO', 'NJ', 'WA', 'AK', 'VA', 'NH', 'UT', 'HI', 'CT', 'MN', 'NY', 'RI', 'DE', 'NV', 'IL', 'TX', 'AZ', 'GA', 'WI', 'OR', 'ND', 'VT', 'FL', 'NE', 'NC', 'PA', 'SD', 'IA', 'MI', 'IN', 'OH', 'SC', 'KS', 'MO', 'TN', 'ME'],
        'med': ['NM', 'ID', 'LA', 'OK', 'MT', 'KY', 'AL', 'WV', 'AR', 'MS'],
        'low': ['WY', 'NM', 'MS', 'WV', 'AR', 'AL', 'KY', 'OK', 'LA', 'SC']
    },
    'prawo_kredytowe': {
        'restrictive': ['PA', 'NY', 'CA', 'AZ', 'HI', 'IN', 'IA', 'KS', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'ND', 'OH', 'OK', 'OR', 'SD', 'TN', 'UT', 'WV', 'WI', 'WY'],
        'moderate': ['DE', 'FL', 'GA', 'ID', 'IL', 'KY', 'ME', 'MD', 'MA', 'MI', 'MN', 'MO', 'NC', 'RI', 'TX', 'VT', 'VA', 'WA'],
        'lenient': ['AK', 'AL', 'AR', 'CO', 'CT', 'DC', 'LA', 'MS', 'SC', 'WI', 'WY', 'SD']
    }
}


In [ ]:
# Tworzymy odwrócone słowniki dla każdej kategorii
położenie_map = {state: region for region, states in slownik['położenie'].items() for state in states}
zamożność_map = {state: zamożność for zamożność, states in slownik['zamożność'].items() for state in states}
prawo_map = {state: prawo for prawo, states in slownik['prawo_kredytowe'].items() for state in states}

# Mapowanie kolumny address_state
data['location'] = data['address_state'].map(położenie_map)
data['wealth'] = data['address_state'].map(zamożność_map)
data['law_regulations'] = data['address_state'].map(prawo_map)

In [ ]:
data[['address_state', 'location', 'wealth', 'law_regulations']]

In [ ]:
data = pd.get_dummies(data, columns=['location', 'wealth', 'law_regulations'], drop_first=True)
data = data.drop('address_state', axis=1)

In [ ]:
top_purpose = data['purpose'].value_counts().nlargest(2).index
data['purpose_grouped'] = data['purpose'].apply(lambda x: x if x in top_purpose else 'other')
data = pd.get_dummies(data, columns=['purpose_grouped'], drop_first=True)

data = data.drop('purpose', axis=1)

In [ ]:
cat_cols = ['verification_status', 'loan_status', 'home_ownership']
data = pd.get_dummies(data, columns=cat_cols, drop_first=True)

bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [ ]:
data['term'] = data['term'].astype(str).str.strip()
data['60_month_term'] = np.where(data['term'] == '60 months', 1, 0)
data = data.drop('term', axis=1)


In [ ]:
data.head()

In [ ]:
cols_to_scale = ['date_diff', 'emp_length']           
scaler = StandardScaler()
data[cols_to_scale] = scaler.fit_transform(data[cols_to_scale])
data = data.drop(['issue_month', 'issue_day'], axis=1)



In [ ]:
data.head()

## Sprawdzanie czy zmienne prawidłowo zestandaryzowane

In [ ]:
means = data.mean(axis=0)
stds = data.std(axis=0)

print("Średnie cech:", means)
print("Odchylenia standardowe cech:", stds)

In [ ]:
data.to_csv('data_after_prep.csv', index=False)